# 04 — Statistical Analysis

Before jumping into machine learning, let's use statistical tests to formally
check whether certain factors are significantly associated with late delivery risk.

This helps build intuition about which features matter and gives us
confidence that the patterns we saw in EDA aren't just random noise.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## Load Data

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")
print(f"Dataset: {df.shape}")

Dataset: (180519, 36)


## 4.1 Chi-Square Test — Shipping Mode vs Late Delivery

**Question:** Is the late delivery rate significantly different across shipping modes,
or could the differences we see be due to random variation?

- **H₀ (Null):** Shipping mode and late delivery risk are independent
- **H₁ (Alternative):** There is a significant association between shipping mode and late delivery
- **Significance level:** α = 0.05

In [3]:
# Create contingency table
contingency = pd.crosstab(df["Shipping Mode"], df["Late_delivery_risk"])
print("Contingency Table:")
print(contingency)
print()

# Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"Degrees of freedom: {dof}")
print(f"\nConclusion: ", end="")
if p_value < 0.05:
    print("Reject H₀ — Shipping mode is significantly associated with late delivery risk.")
    print("The differences in late delivery rates across shipping modes are statistically significant.")
else:
    print("Fail to reject H₀ — No significant association found.")

Contingency Table:
Late_delivery_risk      0      1
Shipping Mode                   
First Class          1301  26513
Same Day             5283   4454
Second Class         8229  26987
Standard Class      66729  41023

Chi-square statistic: 37716.04
p-value: 0.00e+00
Degrees of freedom: 3

Conclusion: Reject H₀ — Shipping mode is significantly associated with late delivery risk.
The differences in late delivery rates across shipping modes are statistically significant.


## 4.2 T-Test — Scheduled Shipping Days: Late vs On-Time

**Question:** Do late deliveries have a different number of scheduled shipping days
compared to on-time deliveries?

- **H₀:** Mean scheduled days is the same for late and on-time deliveries
- **H₁:** Mean scheduled days is different
- **Significance level:** α = 0.05

In [4]:
late = df[df["Late_delivery_risk"] == 1]["Days for shipment (scheduled)"]
on_time = df[df["Late_delivery_risk"] == 0]["Days for shipment (scheduled)"]

print(f"Late deliveries — mean scheduled days: {late.mean():.2f} (std: {late.std():.2f})")
print(f"On-time deliveries — mean scheduled days: {on_time.mean():.2f} (std: {on_time.std():.2f})")

# Welch's t-test (does not assume equal variances)
t_stat, p_value = stats.ttest_ind(late, on_time, equal_var=False)

print(f"\nt-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"\nConclusion: ", end="")
if p_value < 0.05:
    print("Reject H₀ — Scheduled shipping days differ significantly between late and on-time deliveries.")
else:
    print("Fail to reject H₀ — No significant difference found.")

Late deliveries — mean scheduled days: 2.47 (std: 1.37)
On-time deliveries — mean scheduled days: 3.49 (std: 1.15)

t-statistic: -171.73
p-value: 0.00e+00

Conclusion: Reject H₀ — Scheduled shipping days differ significantly between late and on-time deliveries.


## 4.3 ANOVA — Sales Across Customer Segments

**Question:** Does average order value (Sales) vary significantly across
customer segments (Consumer, Corporate, Home Office)?

- **H₀:** Mean sales is the same across all customer segments
- **H₁:** At least one segment has a different mean
- **Significance level:** α = 0.05

In [5]:
segments = df["Customer Segment"].unique()
groups = [df[df["Customer Segment"] == seg]["Sales"].values for seg in segments]

for seg, grp in zip(segments, groups):
    print(f"{seg}: mean=${grp.mean():.2f}, median=${np.median(grp):.2f}, n={len(grp):,}")

f_stat, p_value = stats.f_oneway(*groups)

print(f"\nF-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"\nConclusion: ", end="")
if p_value < 0.05:
    print("Reject H₀ — Sales differ significantly across customer segments.")
else:
    print("Fail to reject H₀ — No significant difference in sales across segments.")

Consumer: mean=$204.22, median=$199.92, n=93,504
Home Office: mean=$202.34, median=$199.92, n=32,226
Corporate: mean=$203.84, median=$199.92, n=54,789

F-statistic: 2.45
p-value: 8.64e-02

Conclusion: Fail to reject H₀ — No significant difference in sales across segments.


## 4.4 Correlation Analysis — Key Numerical Features

**Question:** Which numerical features have the strongest (linear) relationship
with late delivery risk?

Note: A low correlation doesn't mean the feature is useless — it might have
a non-linear relationship that tree-based models can capture.

In [6]:
num_cols = df.select_dtypes(include=np.number).columns
correlations = df[num_cols].corr()["Late_delivery_risk"].drop("Late_delivery_risk")
correlations = correlations.sort_values(key=abs, ascending=False)

print("Correlations with Late_delivery_risk:")
print("=" * 50)
for feat, corr in correlations.items():
    strength = "weak"
    if abs(corr) > 0.3:
        strength = "moderate"
    if abs(corr) > 0.5:
        strength = "strong"
    print(f"  {feat:40s} {corr:+.4f}  ({strength})")

print(f"\nNote: All correlations are quite weak (|r| < 0.3).")
print("This suggests non-linear relationships → tree-based models should be more effective than linear models.")

Correlations with Late_delivery_risk:
  Days for shipment (scheduled)            -0.3694  (moderate)
  Order_Hour                               +0.0470  (weak)
  Order Item Total                         -0.0038  (weak)
  Sales per customer                       -0.0038  (weak)
  Order Profit Per Order                   -0.0037  (weak)
  Benefit per order                        -0.0037  (weak)
  Sales                                    -0.0036  (weak)
  Customer Zipcode                         +0.0031  (weak)
  Order_Month                              +0.0026  (weak)
  Order Item Profit Ratio                  -0.0023  (weak)
  Order_Year                               -0.0022  (weak)
  Order Item Product Price                 -0.0022  (weak)
  Product Price                            -0.0022  (weak)
  Order_Day                                -0.0020  (weak)
  Order Item Cardprod Id                   +0.0015  (weak)
  Order Item Discount                      -0.0007  (weak)
  Order Item D

## 4.5 Point-Biserial Correlation — Order Quantity vs Late Delivery

**Question:** Is there a significant linear relationship between order quantity
and late delivery?

Point-biserial correlation is the appropriate test when one variable is binary
and the other is continuous.

In [7]:
r, p_value = stats.pointbiserialr(
    df["Late_delivery_risk"],
    df["Order Item Quantity"]
)

print(f"Point-biserial correlation: {r:.4f}")
print(f"p-value: {p_value:.2e}")
sig_text = "statistically significant" if p_value < 0.05 else "not significant"
str_text = "weak" if abs(r) < 0.1 else "moderate"
print(f"\nInterpretation: The correlation is {sig_text},")
print(f"but the magnitude ({abs(r):.4f}) suggests a very {str_text} linear relationship.")
print("Order quantity alone is not a strong linear predictor, but may interact with other features.")

Point-biserial correlation: -0.0001
p-value: 9.53e-01

Interpretation: The correlation is not significant,
but the magnitude (0.0001) suggests a very weak linear relationship.
Order quantity alone is not a strong linear predictor, but may interact with other features.


## Summary

| Test | Question | Result |
|------|----------|--------|
| Chi-square | Shipping mode related to late delivery? | Significant — different modes have different delay rates |
| Welch's t-test | Scheduled days differ for late vs on-time? | Significant — late deliveries tend to have different scheduled days |
| ANOVA | Sales differ across customer segments? | Tested — see result above |
| Correlation | Which features linearly relate to target? | All weak — suggests non-linear patterns |
| Point-biserial | Order quantity related to delay? | Very weak linear relationship |

**Key insight:** The weak linear correlations across all features strongly suggest
that a linear model (like Logistic Regression) will underperform compared to
tree-based models that can capture non-linear interactions. This will be confirmed
in the next notebook.

**Next step:** Model training and comparison